# Two numbers, one decision

Take two numbers. If their product is positive, label the example `1`; otherwise label it `0`. For example, `-2 * -3` is positive, so its label is `1`. Our tiny neural network has adjustable weights and practices this rule on generated examples.

Before running: how many rows will be held back if we generate 200 and train on the first 160? **40.** The held-back rows tell us whether the model learned something beyond memorizing its training rows. The code uses PyTorch on the CPU. Read [the network lesson](../../docs/04-neural-networks.md) first.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
torch.manual_seed(42)
features = torch.randn(200, 2)
targets = (features[:, 0] * features[:, 1] > 0).long()
train_x, valid_x = features[:160], features[160:]
train_y, valid_y = targets[:160], targets[160:]
model = torch.nn.Sequential(torch.nn.Linear(2, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = torch.nn.CrossEntropyLoss()
for epoch in range(80):
    model.train()
    optimizer.zero_grad()
    logits = model(train_x)
    loss = criterion(logits, train_y)
    loss.backward()
    optimizer.step()
model.eval()
with torch.no_grad():
    validation_logits = model(valid_x)
    accuracy = (validation_logits.argmax(dim=1) == valid_y).float().mean().item()
assert validation_logits.shape == (40, 2)
assert accuracy > 0.65
print('train loss:', round(loss.item(), 3), 'validation accuracy:', round(accuracy, 3))

## Check
Why did we not fit or tune on the last 40 rows? They represent held-out examples. Try replacing ReLU with the identity function: does the nonlinear boundary remain easy to learn? Add a second independent test split before using this pattern in a real study.